
# Kuroshio Daily R1 FIX3 — PyTorch 2.6+ 安全 Checkpoint 修复版
## RTX 5080 / WSL / VS Code

本版本在 FIX2 基础上修复：

```text
UnpicklingError: Weights only load failed
Unsupported global: numpy._core.multiarray._reconstruct
```

根因是 PyTorch 2.6+ 的 `torch.load()` 默认：

```python
weights_only=True
```

而 FIX2 checkpoint 中保存了 NumPy 数组：

```python
channel_mean = np.ndarray
channel_std  = np.ndarray
```

因此安全反序列化器会拒绝旧 checkpoint。

### FIX3 的处理方式

1. 新 checkpoint 只把 `channel_mean/channel_std` 保存为 **torch.Tensor**；
2. 推理阶段显式使用 `weights_only=True`；
3. 如果检测到旧 FIX2 checkpoint，提供一次性“可信本地 checkpoint 迁移”函数：
   - 仅对你自己本地刚训练生成的 checkpoint 使用 `weights_only=False`
   - 立即转换并重新保存成 FIX3 安全格式
4. 后续所有加载均使用安全格式，不再依赖 NumPy pickle。


In [ ]:

# ===== 0. 可选：首次环境安装 =====
# 建议先按配套的《WSL工作目录配置指南》创建虚拟环境。
# 若当前 kernel 已安装依赖，请保持 RUN_INSTALL=False。

RUN_INSTALL = False

if RUN_INSTALL:
    import sys, subprocess
    pkgs = [
        "numpy", "pandas", "scipy", "xarray", "netCDF4", "h5netcdf",
        "matplotlib", "scikit-learn", "tqdm", "joblib", "pyarrow",
        "zarr<3", "numcodecs", "nbformat"
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *pkgs])
    print("基础依赖安装完成。PyTorch 请按指南单独安装 CUDA wheel。")


In [ ]:
# ===== 1. 唯一需要优先检查的配置区 =====
from pathlib import Path
from dataclasses import dataclass, asdict
import os, re, json, math, random, warnings, hashlib
import numpy as np
import pandas as pd

@dataclass
class CFG:
    PROJECT_ROOT: str = "/mnt/d/KuroshioDailyR1"

    # Windows E:\ -> WSL /mnt/e
    EXTERNAL_DRIVE_ROOT: str = "/mnt/e"
    YEAR_FOLDERS: tuple = ("2023", "2024", "2025")

    START_DATE: str = "2023-09-01"
    END_DATE: str = "2025-03-31"
    NETCDF_SUFFIXES: tuple = (".nc", ".nc4", ".cdf")

    LON_MIN: float = 110.0
    LON_MAX: float = 160.0
    LAT_MIN: float = 10.0
    LAT_MAX: float = 45.0
    TARGET_DEPTH_M: float = 50.0

    # 如果一个文件包含多个时间帧，则求时间平均
    MEAN_MULTI_TIME: bool = True
    AUDIT_FILES_PER_YEAR: int = 2

    # characteristic-line
    SEED_LON_MIN: float = 121.0
    SEED_LON_MAX: float = 122.0
    SEED_LAT_MIN: float = 22.0
    SEED_LAT_MAX: float = 23.0
    TERMINAL_LON: float = 146.0
    EXTENSION_LON: float = 141.0
    AUX_HALF_KM: float = 70.0
    AUX_POINTS: int = 21
    STEP_MAIN_KM: float = 10.0
    STEP_RESCUE_KM: float = 20.0
    STEP_EXTENSION_KM: float = 50.0
    REPEAT_THRESHOLD_KM: float = 1.5
    RESCUE_ANGLE_DEG: float = 30.0
    RESCUE_STEPS: int = 20
    MAX_TRACK_POINTS: int = 1000

    # CNN patch
    PATCH_SIZE: int = 17
    PATCH_CHANNELS: int = 5
    MAX_POS_PER_DAY: int = 24
    NEG_PER_POS: int = 2
    PATCH_STRIDE_ALONG_TRACK: int = 4
    CNN_EPOCHS: int = 25
    CNN_BATCH: int = 256
    CNN_LR: float = 2e-3

    SCORE_W_SPEED: float = 0.50
    SCORE_W_CNN: float = 0.30
    SCORE_W_ALIGN: float = 0.20

    MAX_SHORT_GAP_DAYS_FOR_FUTURE_PATH_INTERP: int = 2

    # FIX2 baseline / weak-label gate
    TRACKER_VERSION: str = "R1_FIX2_EAST_ZERO_V2"
    MIN_WEAK_LABEL_DAYS: int = 20
    MIN_COMPLETION_RATE_FOR_CNN: float = 0.30
    SEED: int = 20260925

cfg = CFG()

ROOT = Path(cfg.PROJECT_ROOT)
USB_ROOT = Path(cfg.EXTERNAL_DRIVE_ROOT)
YEAR_DIRS = [USB_ROOT / y for y in cfg.YEAR_FOLDERS]

DIRS = {
    "cache50": ROOT / "cache_50m",
    "tracks_base": ROOT / "tracks_baseline_FIX2_east0",
    "tracks_cnn": ROOT / "tracks_cnn_FIX2_east0",
    "models": ROOT / "models",
    "reports": ROOT / "reports",
    "figures": ROOT / "figures",
    "dataset": ROOT / "dataset",
}
for p in [ROOT, *DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(cfg.SEED)
np.random.seed(cfg.SEED)

print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))
print("\\nPROJECT_ROOT =", ROOT)
print("USB_ROOT     =", USB_ROOT)

if not USB_ROOT.exists():
    raise FileNotFoundError(
        f"WSL 中未找到 {USB_ROOT}. "
        "先在终端执行 ls /mnt/e；若不存在，可尝试 "
        "sudo mkdir -p /mnt/e && sudo mount -t drvfs E: /mnt/e"
    )

for yd in YEAR_DIRS:
    print(f"{yd}: exists={yd.exists()}")
    if not yd.exists():
        raise FileNotFoundError(f"缺少年度目录: {yd}")


In [ ]:

# ===== 2. 环境 / GPU 检查 =====
import sys, platform, subprocess

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA runtime:", torch.version.cuda)
        cc = torch.cuda.get_device_capability(0)
        print("Compute capability:", cc)
except Exception as e:
    print("[WARN] PyTorch 未就绪:", repr(e))

try:
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=15)
    print(out.stdout[:4000] if out.returncode == 0 else out.stderr[:2000])
except Exception as e:
    print("[WARN] nvidia-smi 调用失败:", repr(e))



## 3. 数据审计策略

旧 MATLAB 流程存在若干容易影响日尺度批处理的问题，本 Notebook 不直接照搬这些实现细节：

- 日期正则在 `uo` 与 `vo` 分支中不一致；
- 旧流程先整块 `ncread` 全球 4D 场，再切 601×421×36，I/O 和内存浪费很大；
- 50 m 插值脚本曾写成 `for i = 389:389`，实际上只处理一个文件；
- `uo` 成功数统计误用了 `vo_*.mat`；
- 原追踪器中 rescue 步长的变量值与论文描述不一致，且有一处经度位移使用了错误变量；
- 日尺度文件缺失不能直接按“月份列号”处理，因此必须以真实日期 manifest 为主键。

本流程采用：**文件扫描 → 日期 manifest → 单日 50m 缓存 → 单日追踪 → 日期化双分支数据集**。


In [ ]:
# ===== 2.5 外置 E 盘目录快速审计 =====
def human_size(n):
    n = float(n)
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024 or u == "TB":
            return f"{n:.2f} {u}"
        n /= 1024

folder_rows = []
for year_dir in YEAR_DIRS:
    files_year = [
        p for p in year_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in cfg.NETCDF_SUFFIXES
    ]
    total_bytes = sum(p.stat().st_size for p in files_year)
    folder_rows.append({
        "year_folder": year_dir.name,
        "path": str(year_dir),
        "netcdf_files": len(files_year),
        "total_size": human_size(total_bytes),
        "first_names": " | ".join([p.name for p in sorted(files_year)[:3]])
    })

folder_audit = pd.DataFrame(folder_rows)
display(folder_audit)

if folder_audit["netcdf_files"].sum() == 0:
    raise RuntimeError(
        "2023/2024/2025 三个目录中没有找到 .nc/.nc4/.cdf 文件。"
        "请检查文件后缀或 E: 是否正确挂载。"
    )


In [ ]:
# ===== 3. 只扫描 E:\2023 / 2024 / 2025，建立真实日期 manifest =====
import xarray as xr
from tqdm.auto import tqdm

DATE_PATTERNS = [
    re.compile(r"mean[_-]?(\d{8})", re.I),
    re.compile(r"_(\d{8})(?:_|\.|-)", re.I),
    re.compile(r"(?<!\d)((?:19|20)\d{6})(?!\d)")
]
REV_PATTERN = re.compile(r"(?:_R|-R|R)(\d{8})", re.I)

def date_from_filename(name: str):
    for pat in DATE_PATTERNS:
        m = pat.search(name)
        if m:
            try:
                return pd.to_datetime(m.group(1), format="%Y%m%d").normalize()
            except Exception:
                pass
    return pd.NaT

def revision_from_filename(name: str):
    m = REV_PATTERN.search(name)
    if not m:
        return pd.NaT
    try:
        return pd.to_datetime(m.group(1), format="%Y%m%d").normalize()
    except Exception:
        return pd.NaT

def date_from_netcdf_time(path: Path):
    # 仅当文件名没有日期时才打开NetCDF，减少外置USB I/O
    try:
        with xr.open_dataset(path, decode_times=True) as ds:
            for tn in ["time", "TIME", "time_counter"]:
                if tn in ds.coords or tn in ds.variables:
                    vals = np.asarray(ds[tn].values).ravel()
                    if len(vals):
                        dt = pd.to_datetime(vals[0])
                        if not pd.isna(dt):
                            return pd.Timestamp(dt).normalize()
    except Exception:
        return pd.NaT
    return pd.NaT

date0 = pd.Timestamp(cfg.START_DATE)
date1 = pd.Timestamp(cfg.END_DATE)

candidate_files = []
for year_dir in YEAR_DIRS:
    for p in year_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in cfg.NETCDF_SUFFIXES:
            candidate_files.append(p)
candidate_files = sorted(candidate_files)

print("三个年度目录共发现 NetCDF:", len(candidate_files))

records = []
unparsed = []
for p in candidate_files:
    dt = date_from_filename(p.name)
    parse_method = "filename"
    if pd.isna(dt):
        dt = date_from_netcdf_time(p)
        parse_method = "netcdf_time" if not pd.isna(dt) else "failed"
    rev = revision_from_filename(p.name)
    st = p.stat()
    source_year_folder = next((y for y in cfg.YEAR_FOLDERS if (USB_ROOT / y) in p.parents), p.parent.name)
    rec = {
        "date": dt,
        "path": str(p),
        "source_year_folder": source_year_folder,
        "filename": p.name,
        "parse_method": parse_method,
        "revision_date": rev,
        "size_bytes": int(st.st_size),
        "mtime_ns": int(st.st_mtime_ns),
    }
    records.append(rec)
    if pd.isna(dt):
        unparsed.append(rec)

scan = pd.DataFrame(records)
valid_scan = scan.dropna(subset=["date"]).copy()
valid_scan = valid_scan[
    (valid_scan["date"] >= date0) &
    (valid_scan["date"] <= date1)
].copy()

valid_scan["date_year"] = valid_scan["date"].dt.year.astype(str)
valid_scan["year_folder_mismatch"] = (
    valid_scan["source_year_folder"].astype(str) != valid_scan["date_year"]
)

dup_count = valid_scan.groupby("date").size().rename("candidate_count")
valid_scan["revision_rank"] = valid_scan["revision_date"].fillna(pd.Timestamp("1900-01-01"))

chosen = (
    valid_scan.sort_values(
        ["date", "revision_rank", "size_bytes", "mtime_ns", "filename"],
        ascending=[True, False, False, False, False]
    )
    .drop_duplicates("date", keep="first")
    [["date","path","source_year_folder","filename","parse_method",
      "revision_date","size_bytes","year_folder_mismatch"]]
)

expected = pd.DataFrame({"date": pd.date_range(date0, date1, freq="D")})
manifest = expected.merge(chosen, on="date", how="left")
manifest = manifest.merge(dup_count, on="date", how="left")
manifest["candidate_count"] = manifest["candidate_count"].fillna(0).astype(int)
manifest["raw_status"] = np.where(manifest["path"].notna(), "present", "missing")
manifest["cache_status"] = "pending"
manifest["track_status"] = "pending"

manifest_path = DIRS["reports"] / "daily_manifest.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
scan.to_csv(DIRS["reports"] / "all_scanned_netcdf.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(unparsed).to_csv(DIRS["reports"] / "unparsed_netcdf.csv", index=False, encoding="utf-8-sig")

print("\\n===== 日期覆盖审计 =====")
print("理论日期数:", len(expected))
print("实际有文件:", int((manifest.raw_status == "present").sum()))
print("缺失天数:", int((manifest.raw_status == "missing").sum()))
print("存在重复候选的日期:", int((manifest.candidate_count > 1).sum()))
print("年度目录与文件日期年份不一致:", int(manifest["year_folder_mismatch"].fillna(False).sum()))

year_summary = (
    manifest.assign(year=manifest["date"].dt.year)
    .groupby("year")
    .agg(expected_days=("date","size"),
         present_days=("raw_status",lambda s:(s=="present").sum()),
         missing_days=("raw_status",lambda s:(s=="missing").sum()),
         duplicate_days=("candidate_count",lambda s:(s>1).sum()))
)
display(year_summary)

print("\\n前20个缺失日期：")
display(manifest.loc[manifest.raw_status == "missing", ["date","raw_status","candidate_count"]].head(20))

print("\\n重复日期示例：")
display(manifest.loc[manifest.candidate_count > 1, ["date","filename","candidate_count","revision_date"]].head(20))
print("manifest ->", manifest_path)


In [ ]:
# ===== 4. NetCDF结构审计 + 直接裁剪研究区并插值到50m =====
NAME_CANDIDATES = {
    "u": ["uo", "u", "eastward_sea_water_velocity"],
    "v": ["vo", "v", "northward_sea_water_velocity"],
    "lon": ["longitude", "lon", "x"],
    "lat": ["latitude", "lat", "y"],
    "depth": ["depth", "deptht", "lev", "z"],
    "time": ["time", "TIME", "time_counter"],
}

def pick_name(obj, candidates, kind):
    all_names = list(dict.fromkeys(list(obj.variables) + list(obj.coords) + list(obj.dims)))
    for c in candidates:
        if c in all_names:
            return c
    raise KeyError(f"无法找到 {kind}; candidates={candidates}; names={all_names[:100]}")

def inspect_netcdf(path: Path):
    info = {"path": str(path), "ok": False}
    try:
        with xr.open_dataset(path, decode_times=True) as ds:
            u_name = pick_name(ds, NAME_CANDIDATES["u"], "u")
            v_name = pick_name(ds, NAME_CANDIDATES["v"], "v")
            lon_name = pick_name(ds, NAME_CANDIDATES["lon"], "lon")
            lat_name = pick_name(ds, NAME_CANDIDATES["lat"], "lat")
            dep_name = pick_name(ds, NAME_CANDIDATES["depth"], "depth")
            time_name = next((t for t in NAME_CANDIDATES["time"] if t in ds.dims or t in ds.coords), None)
            info.update({
                "ok": True,
                "u":u_name, "v":v_name, "lon":lon_name, "lat":lat_name,
                "depth":dep_name, "time":time_name,
                "u_dims":str(ds[u_name].dims), "u_shape":str(ds[u_name].shape),
                "v_dims":str(ds[v_name].dims), "v_shape":str(ds[v_name].shape),
                "lon_min":float(np.nanmin(ds[lon_name].values)),
                "lon_max":float(np.nanmax(ds[lon_name].values)),
                "lat_min":float(np.nanmin(ds[lat_name].values)),
                "lat_max":float(np.nanmax(ds[lat_name].values)),
                "depth_min":float(np.nanmin(ds[dep_name].values)),
                "depth_max":float(np.nanmax(ds[dep_name].values)),
            })
            if time_name:
                info["time_size"] = int(ds.sizes.get(time_name, 1))
    except Exception as e:
        info["error"] = repr(e)
    return info

audit_paths = []
for y in cfg.YEAR_FOLDERS:
    q = manifest[
        (manifest.raw_status == "present") &
        (manifest.date.dt.year == int(y))
    ]["path"].dropna().tolist()
    audit_paths += q[:cfg.AUDIT_FILES_PER_YEAR]

audit_rows = [inspect_netcdf(Path(p)) for p in audit_paths]
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(DIRS["reports"] / "netcdf_structure_audit.csv", index=False, encoding="utf-8-sig")
display(audit_df)

if len(audit_df) and not audit_df["ok"].all():
    raise RuntimeError(
        "NetCDF结构预审计存在失败文件。先检查 reports/netcdf_structure_audit.csv，"
        "不要直接批处理全部USB数据。"
    )

def extract_50m_to_npz(nc_path: Path, out_path: Path, overwrite=False):
    if out_path.exists() and not overwrite:
        return "cached"

    with xr.open_dataset(nc_path, decode_times=True, cache=False) as ds:
        u_name = pick_name(ds, NAME_CANDIDATES["u"], "u")
        v_name = pick_name(ds, NAME_CANDIDATES["v"], "v")
        lon_name = pick_name(ds, NAME_CANDIDATES["lon"], "lon")
        lat_name = pick_name(ds, NAME_CANDIDATES["lat"], "lat")
        dep_name = pick_name(ds, NAME_CANDIDATES["depth"], "depth")

        sub = ds[[u_name, v_name]]

        for cname in [lon_name, lat_name, dep_name]:
            if cname in sub.coords and sub[cname].ndim == 1:
                vals = np.asarray(sub[cname].values)
                if len(vals) > 1 and vals[0] > vals[-1]:
                    sub = sub.sortby(cname)

        sub = sub.sel({
            lon_name: slice(cfg.LON_MIN, cfg.LON_MAX),
            lat_name: slice(cfg.LAT_MIN, cfg.LAT_MAX)
        })

        if sub.sizes.get(lon_name, 0) == 0 or sub.sizes.get(lat_name, 0) == 0:
            raise RuntimeError(
                f"裁剪后经纬度为空: {nc_path.name}; 请检查经度定义是否为0-360或-180~180。"
            )

        for tname in NAME_CANDIDATES["time"]:
            if tname in sub.dims:
                nt = int(sub.sizes[tname])
                if nt > 1 and cfg.MEAN_MULTI_TIME:
                    sub = sub.mean(dim=tname, skipna=True)
                else:
                    sub = sub.isel({tname: 0})
                break

        dep_vals = np.asarray(sub[dep_name].values, dtype=float)
        if not (np.nanmin(dep_vals) <= cfg.TARGET_DEPTH_M <= np.nanmax(dep_vals)):
            raise RuntimeError(
                f"{nc_path.name}: depth范围 {np.nanmin(dep_vals)}~{np.nanmax(dep_vals)} m "
                f"不包含目标 {cfg.TARGET_DEPTH_M} m"
            )

        sub50 = sub.interp({dep_name: cfg.TARGET_DEPTH_M})
        u_da = sub50[u_name].squeeze(drop=True).transpose(lat_name, lon_name)
        v_da = sub50[v_name].squeeze(drop=True).transpose(lat_name, lon_name)

        lon = np.asarray(sub50[lon_name].values, dtype=np.float64)
        lat = np.asarray(sub50[lat_name].values, dtype=np.float64)
        u = np.asarray(u_da.load().values, dtype=np.float32)
        v = np.asarray(v_da.load().values, dtype=np.float32)

        if u.shape != (len(lat), len(lon)) or v.shape != u.shape:
            raise RuntimeError(
                f"异常二维形状: u={u.shape}, v={v.shape}, lat={len(lat)}, lon={len(lon)}"
            )

        finite = np.isfinite(u) & np.isfinite(v)
        if finite.mean() < 0.10:
            raise RuntimeError(f"{nc_path.name}: 有效海流格点比例过低 {finite.mean():.3f}")

        out_path.parent.mkdir(parents=True, exist_ok=True)
        tmp_path = out_path.with_suffix(".tmp.npz")
        np.savez_compressed(
            tmp_path,
            lon=lon, lat=lat, u=u, v=v,
            source_file=np.array(str(nc_path)),
            target_depth_m=np.array(cfg.TARGET_DEPTH_M, dtype=np.float32),
        )
        os.replace(tmp_path, out_path)

    return "written"

first_present = manifest.loc[manifest.raw_status == "present"].iloc[0]
sample_npz = DIRS["cache50"] / f"{first_present.date:%Y%m%d}.npz"
status = extract_50m_to_npz(Path(first_present.path), sample_npz, overwrite=False)

print("sample:", status, sample_npz)
z = np.load(sample_npz)
print("lon:", z["lon"].shape, z["lon"][0], "->", z["lon"][-1])
print("lat:", z["lat"].shape, z["lat"][0], "->", z["lat"][-1])
print("u/v:", z["u"].shape, z["v"].shape)
print("finite u:", np.isfinite(z["u"]).mean())
print("finite v:", np.isfinite(z["v"]).mean())


In [ ]:
# ===== 5. 批量生成50m缓存：按日期顺序、断点续跑、失败可重试 =====
rows = []
present_manifest = manifest[manifest.raw_status == "present"].sort_values("date").copy()

print("待处理存在文件日期:", len(present_manifest))
print("输出缓存目录:", DIRS["cache50"])

for row in tqdm(present_manifest.itertuples(index=False), total=len(present_manifest), desc="E: -> D: cache 50m"):
    dt = pd.Timestamp(row.date)
    out = DIRS["cache50"] / f"{dt:%Y%m%d}.npz"
    try:
        st = extract_50m_to_npz(Path(row.path), out, overwrite=False)
        zz = np.load(out)
        if not (zz["u"].ndim == 2 and zz["v"].shape == zz["u"].shape):
            raise RuntimeError(f"cache shape invalid: {out}")
        rows.append({"date":dt, "cache_status_new":st, "cache_path":str(out), "error":""})
    except Exception as e:
        rows.append({"date":dt, "cache_status_new":"error", "cache_path":str(out), "error":repr(e)})

cache_map = pd.DataFrame(rows)
manifest2 = manifest.drop(columns=["cache_status"]).merge(cache_map, on="date", how="left")
manifest2["cache_status"] = np.where(
    manifest2.raw_status == "missing",
    "missing_raw",
    manifest2["cache_status_new"].fillna("pending")
)
manifest2 = manifest2.drop(columns=["cache_status_new"])
manifest = manifest2
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")

cache_fail = manifest[manifest.cache_status == "error"].copy()
cache_fail.to_csv(DIRS["reports"] / "cache_failures.csv", index=False, encoding="utf-8-sig")

print("\\n===== 50m缓存结果 =====")
print(manifest["cache_status"].value_counts(dropna=False))
if len(cache_fail):
    print("\\n失败文件：")
    display(cache_fail[["date","filename","path","error"]].head(30))
else:
    print("没有缓存失败文件。")


In [ ]:

# ===== 6. 几何与物理特征函数 — FIX2统一“东向=0”的角度约定 =====
from scipy.interpolate import RegularGridInterpolator

EARTH_R_KM = 6371.0088

def wrap_pi(a):
    return (a + np.pi) % (2*np.pi) - np.pi

def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.asarray, (lon1, lat1, lon2, lat2))
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    dl = np.deg2rad(lon2-lon1)
    dp = p2-p1
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*EARTH_R_KM*np.arcsin(np.minimum(1.0, np.sqrt(a)))

def angle_east_rad(lon1, lat1, lon2, lat2):
    """
    返回与 MATLAB atan2(dy,dx) 一致的方向角：
      0       = 正东
      pi/2    = 正北
      pi/-pi  = 正西
      -pi/2   = 正南

    内部先算标准导航方位角(0=北, pi/2=东)，再转换为东向零角。
    """
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    dl = np.deg2rad(lon2-lon1)
    y = np.sin(dl)*np.cos(p2)
    x = np.cos(p1)*np.sin(p2) - np.sin(p1)*np.cos(p2)*np.cos(dl)
    nav_bearing = np.arctan2(y, x)          # 0=北, +pi/2=东
    east_angle = np.pi/2 - nav_bearing      # 0=东, +pi/2=北
    return wrap_pi(east_angle)

def forward_point(lon, lat, east_angle, distance_km):
    """
    沿 east_angle 前进 distance_km。
    输入 east_angle 与原 MATLAB theta=atan2(v,u) 完全同一约定。
    """
    # spherical forward公式使用导航方位角: 0=北, pi/2=东
    nav_bearing = np.pi/2 - east_angle

    phi1, lam1 = np.deg2rad(lat), np.deg2rad(lon)
    d = distance_km / EARTH_R_KM
    phi2 = np.arcsin(
        np.sin(phi1)*np.cos(d) +
        np.cos(phi1)*np.sin(d)*np.cos(nav_bearing)
    )
    lam2 = lam1 + np.arctan2(
        np.sin(nav_bearing)*np.sin(d)*np.cos(phi1),
        np.cos(d)-np.sin(phi1)*np.sin(phi2)
    )
    return float(np.rad2deg(lam2)), float(np.rad2deg(phi2))

def aux_line(lon, lat, flow_dir, half_km=None, n=None):
    """
    沿 flow_dir 的法线构造140 km辅助线。
    flow_dir: 0=东, pi/2=北。
    """
    half_km = cfg.AUX_HALF_KM if half_km is None else half_km
    n = cfg.AUX_POINTS if n is None else n
    ds = np.linspace(-half_km, half_km, n)
    xs, ys = [], []
    for s in ds:
        theta_perp = flow_dir + (np.pi/2 if s >= 0 else -np.pi/2)
        x, y = forward_point(lon, lat, theta_perp, abs(float(s)))
        xs.append(x); ys.append(y)
    return np.asarray(xs), np.asarray(ys), ds

def matlab_aux_weights(n):
    """
    复现原 MATLAB：
    w(k)=1-abs((n+1)/2-k)/(n-1), k=1..n
    n=11时端点0.5、中心1.0；n=21同样端点0.5、中心1.0。
    """
    k = np.arange(1, n+1, dtype=np.float64)
    return 1.0 - np.abs((n+1)/2.0 - k)/(n-1)

def make_interpolators(lon, lat, u, v):
    return (
        RegularGridInterpolator((lat, lon), u, bounds_error=False, fill_value=np.nan),
        RegularGridInterpolator((lat, lon), v, bounds_error=False, fill_value=np.nan)
    )

def interp_uv(ui, vi, lons, lats):
    pts = np.column_stack([lats, lons])
    return ui(pts), vi(pts)

def metric_features(lon, lat, u, v):
    # array shape [lat, lon]
    speed = np.hypot(u, v)
    latr = np.deg2rad(lat)
    dlat_m = np.gradient(latr) * EARTH_R_KM * 1000.0
    lonr = np.deg2rad(lon)
    dlon_rad = np.gradient(lonr)

    dx = (EARTH_R_KM*1000.0) * np.cos(latr)[:,None] * dlon_rad[None,:]
    dy = dlat_m[:,None]

    du_dy_idx, du_dx_idx = np.gradient(u)
    dv_dy_idx, dv_dx_idx = np.gradient(v)
    with np.errstate(invalid="ignore", divide="ignore"):
        du_dx = du_dx_idx / dx
        dv_dx = dv_dx_idx / dx
        du_dy = du_dy_idx / dy
        dv_dy = dv_dy_idx / dy

    zeta = dv_dx - du_dy
    sn = du_dx - dv_dy
    ss = dv_dx + du_dy
    ow = sn**2 + ss**2 - zeta**2
    return speed.astype(np.float32), zeta.astype(np.float32), ow.astype(np.float32)

def nearest_index(arr, x):
    i = int(np.searchsorted(arr, x))
    if i <= 0:
        return 0
    if i >= len(arr):
        return len(arr)-1
    return i if abs(arr[i]-x) < abs(arr[i-1]-x) else i-1

def extract_patch(features_chw, lon, lat, lon0, lat0, patch_size=None):
    patch_size = cfg.PATCH_SIZE if patch_size is None else patch_size
    r = patch_size//2
    ix, iy = nearest_index(lon, lon0), nearest_index(lat, lat0)

    if iy-r < 0 or iy+r >= len(lat) or ix-r < 0 or ix+r >= len(lon):
        return None

    p = features_chw[:, iy-r:iy+r+1, ix-r:ix+r+1]
    if p.shape[1:] != (patch_size, patch_size):
        return None
    if np.isfinite(p).mean() < 0.90:
        return None

    p = p.copy()
    for c in range(p.shape[0]):
        med = np.nanmedian(p[c])
        if not np.isfinite(med):
            return None
        p[c] = np.nan_to_num(p[c], nan=med, posinf=med, neginf=med)
    return p.astype(np.float32)

# ---- 极小单元测试：防止以后再次混用角度约定 ----
xE, yE = forward_point(130.0, 30.0, 0.0, 10.0)
xN, yN = forward_point(130.0, 30.0, np.pi/2, 10.0)
assert xE > 130.0 and abs(yE-30.0) < 0.05, (xE, yE)
assert yN > 30.0 and abs(xN-130.0) < 0.05, (xN, yN)
print("[FIX2 geometry smoke] east step:", xE, yE)
print("[FIX2 geometry smoke] north step:", xN, yN)
print("[FIX2 geometry smoke] PASS")


In [ ]:

# ===== 7. Adaptive characteristic-line tracker — FIX2 =====
def _seed_point(lon, lat, u, v):
    ix = np.where((lon >= cfg.SEED_LON_MIN) & (lon <= cfg.SEED_LON_MAX))[0]
    iy = np.where((lat >= cfg.SEED_LAT_MIN) & (lat <= cfg.SEED_LAT_MAX))[0]

    if len(ix)==0 or len(iy)==0:
        raise RuntimeError("seed区域不在当前网格内")

    sub_u = u[np.ix_(iy, ix)]
    sub_v = v[np.ix_(iy, ix)]
    sp2 = sub_u**2 + sub_v**2

    if not np.isfinite(sp2).any():
        raise RuntimeError("seed 区域无有效流速")

    j, i = np.unravel_index(np.nanargmax(sp2), sp2.shape)
    lon0, lat0 = float(lon[ix[i]]), float(lat[iy[j]])

    # 与原MATLAB完全一致：0=东，pi/2=北
    theta = float(np.arctan2(sub_v[j,i], sub_u[j,i]))
    return lon0, lat0, theta

def _candidate_select(
    ui, vi,
    center_lon, center_lat,
    mean_dir,
    lon_grid, lat_grid,
    features=None,
    cnn_bundle=None,
    rescue=False,
    prev_axis_lon=None,
    prev_axis_lat=None
):
    # 用mean_dir重新构造真正的“新副线”
    xl, yl, _ = aux_line(center_lon, center_lat, mean_dir)
    uc, vc = interp_uv(ui, vi, xl, yl)
    speed = np.hypot(uc, vc)

    valid = np.isfinite(speed) & np.isfinite(uc) & np.isfinite(vc)
    if not valid.any():
        return None

    local_flow_dir = np.arctan2(vc, uc)  # 0=东
    flow_align = (np.cos(wrap_pi(local_flow_dir - mean_dir)) + 1.0)/2.0

    # FIX2：rescue规则忠实于原MATLAB
    # 原代码比较的是“上一流轴点 -> 候选点”的几何方向与mean_direction之差，
    # 不是候选点自己的局地流速方向。
    if rescue and prev_axis_lon is not None and prev_axis_lat is not None:
        geom_dirs = np.array([
            angle_east_rad(prev_axis_lon, prev_axis_lat, x, y)
            for x,y in zip(xl,yl)
        ])
        geom_diff = np.abs(wrap_pi(geom_dirs - mean_dir))
        valid_rescue = valid & (geom_diff < np.deg2rad(cfg.RESCUE_ANGLE_DEG))

        if valid_rescue.any():
            valid = valid_rescue
        else:
            # 对应旧MATLAB“没有<30°则取最接近者”的思想：
            # 不直接整条线回退为最大流速，而限制到最接近方向的有效点。
            good = np.where(valid)[0]
            if len(good):
                k0 = good[np.argmin(geom_diff[good])]
                valid[:] = False
                valid[k0] = True

    finite_s = speed[valid]
    lo, hi = np.nanmin(finite_s), np.nanmax(finite_s)
    speed_norm = np.zeros_like(speed, dtype=np.float64)
    speed_norm[valid] = (speed[valid]-lo)/(hi-lo+1e-8)

    if cnn_bundle is None:
        # baseline严格保持 characteristic-line 最大流速选择
        score = speed_norm.copy()
        probs = np.full_like(score, np.nan, dtype=np.float32)
    else:
        model, ch_mean, ch_std, device = cnn_bundle
        probs = np.zeros_like(score if 'score' in locals() else speed_norm, dtype=np.float32)

        patches, ids = [], []
        for k in np.where(valid)[0]:
            p = extract_patch(features, lon_grid, lat_grid, xl[k], yl[k])
            if p is not None:
                p = (p - ch_mean[:,None,None])/(ch_std[:,None,None] + 1e-6)
                patches.append(p)
                ids.append(k)

        if patches:
            import torch
            xb = torch.from_numpy(np.stack(patches)).float().to(device)
            with torch.no_grad():
                pp = torch.sigmoid(model(xb)).detach().cpu().numpy().ravel()
            for k,pv in zip(ids,pp):
                probs[k] = pv

        score = (
            cfg.SCORE_W_SPEED*speed_norm +
            cfg.SCORE_W_CNN*probs +
            cfg.SCORE_W_ALIGN*flow_align
        )

    score = np.asarray(score, dtype=np.float64)
    score[~valid] = -np.inf

    if not np.isfinite(score).any():
        return None

    k = int(np.argmax(score))
    return {
        "lon": float(xl[k]),
        "lat": float(yl[k]),
        "u": float(uc[k]),
        "v": float(vc[k]),
        "speed": float(speed[k]),
        "align": float(flow_align[k]),
        "cnn_prob": float(probs[k]) if np.isfinite(probs[k]) else np.nan,
        "score": float(score[k]),
    }

def track_axis(lon, lat, u, v, cnn_bundle=None):
    ui, vi = make_interpolators(lon, lat, u, v)

    speed_field, zeta, ow = metric_features(lon, lat, u, v)
    features = np.stack([u, v, speed_field, zeta, ow], axis=0).astype(np.float32)

    lon0, lat0, theta0 = _seed_point(lon, lat, u, v)

    path_lon, path_lat, dirs = [], [], []
    loop_events = 0
    rescue_remaining = 0

    center_lon, center_lat = lon0, lat0
    current_dir = theta0

    stop_reason = "max_points"

    for step_idx in range(cfg.MAX_TRACK_POINTS):
        # (1) 先沿上一方向构造辅助线
        xl, yl, _ = aux_line(center_lon, center_lat, current_dir)
        uu, vv = interp_uv(ui, vi, xl, yl)

        valid = np.isfinite(uu) & np.isfinite(vv)
        if not valid.any():
            stop_reason = "no_valid_aux_flow"
            break

        # (2) 原MATLAB加权平均流向，权重端点约0.5、中心1.0
        w = matlab_aux_weights(len(xl))
        den = np.sum(w[valid])
        if den <= 0:
            stop_reason = "bad_aux_weight"
            break

        mean_u = np.sum(uu[valid]*w[valid])/den
        mean_v = np.sum(vv[valid]*w[valid])/den
        mean_dir = float(np.arctan2(mean_v, mean_u))  # 0=东

        prev_lon = path_lon[-1] if len(path_lon) else lon0
        prev_lat = path_lat[-1] if len(path_lat) else lat0

        # (3) 按新平均方向构造第二条辅助线并选流轴点
        cand = _candidate_select(
            ui, vi,
            center_lon, center_lat,
            mean_dir,
            lon, lat,
            features=features,
            cnn_bundle=cnn_bundle,
            rescue=(rescue_remaining > 0),
            prev_axis_lon=prev_lon,
            prev_axis_lat=prev_lat
        )

        if cand is None:
            stop_reason = "no_candidate"
            break

        cl, ca = cand["lon"], cand["lat"]

        # (4) 自循环检测：忽略最近5个点，避免自然邻近误判
        repeat_idx = None
        if len(path_lon) > 8:
            hist_lon = np.asarray(path_lon[:-5])
            hist_lat = np.asarray(path_lat[:-5])
            d = haversine_km(hist_lon, hist_lat, cl, ca)
            if len(d) and np.isfinite(d).any() and np.nanmin(d) < cfg.REPEAT_THRESHOLD_KM:
                repeat_idx = int(np.nanargmin(d))

        if repeat_idx is not None:
            loop_events += 1

            # rollback到首次重复的稳定位置
            path_lon = path_lon[:repeat_idx+1]
            path_lat = path_lat[:repeat_idx+1]
            dirs = dirs[:repeat_idx+1]

            if loop_events >= 3 or len(path_lon) < 2:
                stop_reason = "too_many_loops"
                break

            current_dir = dirs[-1]

            # 与论文/旧MATLAB意图一致：主流区陷涡后20 km，KE仍50 km
            step_km = (
                cfg.STEP_EXTENSION_KM
                if path_lon[-1] > cfg.EXTENSION_LON
                else cfg.STEP_RESCUE_KM
            )
            center_lon, center_lat = forward_point(
                path_lon[-1], path_lat[-1], current_dir, step_km
            )
            rescue_remaining = cfg.RESCUE_STEPS
            continue

        path_lon.append(cl)
        path_lat.append(ca)
        dirs.append(mean_dir)

        if cl >= cfg.TERMINAL_LON:
            stop_reason = "terminal_lon"
            break

        # (5) 正常向下游推进
        step_km = (
            cfg.STEP_EXTENSION_KM
            if cl > cfg.EXTENSION_LON
            else cfg.STEP_MAIN_KM
        )
        center_lon, center_lat = forward_point(cl, ca, mean_dir, step_km)
        current_dir = mean_dir
        rescue_remaining = max(0, rescue_remaining-1)

        # 明显跑出研究域时及时终止，避免无限乱跑
        if not (cfg.LON_MIN-1 <= center_lon <= cfg.LON_MAX+1 and
                cfg.LAT_MIN-1 <= center_lat <= cfg.LAT_MAX+1):
            stop_reason = "left_domain"
            break

    completed = bool(
        len(path_lon) > 2 and
        np.isfinite(path_lon).any() and
        np.nanmax(path_lon) >= cfg.TERMINAL_LON
    )

    return {
        "lon": np.asarray(path_lon, dtype=np.float32),
        "lat": np.asarray(path_lat, dtype=np.float32),
        "direction": np.asarray(dirs, dtype=np.float32),
        "completed": completed,
        "loop_events": int(loop_events),
        "n_points": int(len(path_lon)),
        "stop_reason": stop_reason,
        "tracker_version": cfg.TRACKER_VERSION,
    }

def track_geometry_metrics(track):
    x, y = np.asarray(track["lon"]), np.asarray(track["lat"])

    if len(x) == 0:
        return {
            "path_km":np.nan,
            "p95_turn_deg":np.nan,
            "max_turn_deg":np.nan,
            "max_lon":np.nan,
            "final_lon":np.nan,
            "final_lat":np.nan,
            "lon_progress_deg":np.nan,
        }

    base = {
        "max_lon":float(np.nanmax(x)),
        "final_lon":float(x[-1]),
        "final_lat":float(y[-1]),
        "lon_progress_deg":float(np.nanmax(x)-x[0]),
    }

    if len(x) < 5:
        return {
            **base,
            "path_km":float(np.nansum(haversine_km(x[:-1],y[:-1],x[1:],y[1:]))) if len(x)>1 else 0.0,
            "p95_turn_deg":np.nan,
            "max_turn_deg":np.nan,
        }

    seg = haversine_km(x[:-1], y[:-1], x[1:], y[1:])
    b = np.array([
        angle_east_rad(x[i],y[i],x[i+1],y[i+1])
        for i in range(len(x)-1)
    ])
    turn = np.abs(np.rad2deg(wrap_pi(np.diff(b))))

    return {
        **base,
        "path_km":float(np.nansum(seg)),
        "p95_turn_deg":float(np.nanpercentile(turn,95)) if len(turn) else np.nan,
        "max_turn_deg":float(np.nanmax(turn)) if len(turn) else np.nan,
    }

def save_track(track, path):
    np.savez_compressed(
        path,
        lon=track["lon"],
        lat=track["lat"],
        direction=track["direction"],
        completed=np.array(track["completed"]),
        loop_events=np.array(track["loop_events"]),
        n_points=np.array(track["n_points"]),
        stop_reason=np.array(track["stop_reason"]),
        tracker_version=np.array(track["tracker_version"]),
    )


In [ ]:

# ===== 8. FIX2 baseline tracker：使用新cache目录，绝不复用FIX1错误轨迹 =====
track_rows = []

cache_ok = manifest[
    manifest.cache_status.isin(["cached","written"])
].copy().sort_values("date")

print("FIX2 tracker version:", cfg.TRACKER_VERSION)
print("50m可用日期:", len(cache_ok))
print("FIX2 baseline输出:", DIRS["tracks_base"])

for row in tqdm(
    cache_ok.itertuples(index=False),
    total=len(cache_ok),
    desc="FIX2 baseline axis"
):
    dt = pd.Timestamp(row.date)
    out = DIRS["tracks_base"] / f"{dt:%Y%m%d}.npz"

    try:
        if out.exists():
            zz = np.load(out)
            old_ver = str(zz["tracker_version"].item()) if "tracker_version" in zz.files else ""
            if old_ver == cfg.TRACKER_VERSION:
                tr = {
                    "lon":zz["lon"],
                    "lat":zz["lat"],
                    "direction":zz["direction"],
                    "completed":bool(zz["completed"]),
                    "loop_events":int(zz["loop_events"]),
                    "n_points":int(zz["n_points"]),
                    "stop_reason":str(zz["stop_reason"].item()) if "stop_reason" in zz.files else "unknown",
                    "tracker_version":old_ver,
                }
                status = "cached"
            else:
                z = np.load(DIRS["cache50"] / f"{dt:%Y%m%d}.npz")
                tr = track_axis(z["lon"],z["lat"],z["u"],z["v"],cnn_bundle=None)
                save_track(tr,out)
                status = "rewritten_version_mismatch"
        else:
            z = np.load(DIRS["cache50"] / f"{dt:%Y%m%d}.npz")
            tr = track_axis(z["lon"],z["lat"],z["u"],z["v"],cnn_bundle=None)
            save_track(tr,out)
            status = "written"

        gm = track_geometry_metrics(tr)

        track_rows.append({
            "date":dt,
            "status":status,
            "completed":tr["completed"],
            "loop_events":tr["loop_events"],
            "n_points":tr["n_points"],
            "stop_reason":tr["stop_reason"],
            **gm
        })

    except Exception as e:
        track_rows.append({
            "date":dt,
            "status":"error",
            "completed":False,
            "loop_events":np.nan,
            "n_points":0,
            "stop_reason":"exception",
            "path_km":np.nan,
            "p95_turn_deg":np.nan,
            "max_turn_deg":np.nan,
            "max_lon":np.nan,
            "final_lon":np.nan,
            "final_lat":np.nan,
            "lon_progress_deg":np.nan,
            "error":repr(e),
        })

baseline_report = pd.DataFrame(track_rows).sort_values("date")
baseline_report.to_csv(
    DIRS["reports"]/"baseline_tracker_report_FIX2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n===== FIX2 baseline核心统计 =====")
print("days =", len(baseline_report))
print("completion =", baseline_report.completed.mean())
print("days with >=1 loop =", (baseline_report.loop_events.fillna(0)>0).mean())

print("\nstop_reason:")
display(baseline_report["stop_reason"].value_counts(dropna=False).to_frame("count"))

print("\nloop_events:")
display(baseline_report["loop_events"].value_counts(dropna=False).sort_index().to_frame("count"))

print("\nmax_lon quantiles:")
display(baseline_report["max_lon"].describe(percentiles=[.1,.25,.5,.75,.9,.95]))

print("\n最差进展样本：")
display(
    baseline_report.sort_values(["max_lon","n_points"])
    [["date","completed","loop_events","n_points","max_lon","final_lon",
      "final_lat","stop_reason","p95_turn_deg"]]
    .head(20)
)

# 这个文件是后续定位失败原因最重要的diagnostic
baseline_report.to_csv(
    DIRS["reports"]/"baseline_failure_diagnostic_FIX2.csv",
    index=False,
    encoding="utf-8-sig"
)



## 9. Anti-Eddy CNN 弱监督标签 — FIX2 Quality Gate

FIX1 的条件：

```python
completed & (loop_events == 0)
```

过于严格，而且在 tracker 几何方向出错时会直接得到 0 天，随后 `np.stack([])` 才报错。

FIX2 分成三级：

- **Tier A**：完整到达 146°E、0 次 loop；
- **Tier B**：完整到达、≤1 次 loop，并且路径转向不过度异常；
- **Tier C**：完整到达、≤2 次 loop，然后按几何质量排序选较稳定日期。

原则是：**只有完整到达终点且几何可接受的最终路径才允许产生弱监督正样本。**
如果连 Tier C 都不足，Notebook 会停止 CNN 训练并要求先修 baseline，不会制造伪训练数据。


In [ ]:

# ===== 9. 构建弱监督 patch 数据集 — FIX2质量Gate =====
def load_track_npz(path):
    z = np.load(path)
    return {
        "lon":z["lon"],
        "lat":z["lat"],
        "direction":z["direction"],
        "completed":bool(z["completed"]),
        "loop_events":int(z["loop_events"]),
        "n_points":int(z["n_points"]),
        "stop_reason":str(z["stop_reason"].item()) if "stop_reason" in z.files else "unknown",
    }

def day_features(dt):
    z = np.load(DIRS["cache50"] / f"{pd.Timestamp(dt):%Y%m%d}.npz")
    sp, ze, ow = metric_features(z["lon"],z["lat"],z["u"],z["v"])
    feat = np.stack(
        [z["u"],z["v"],sp,ze,ow],
        axis=0
    ).astype(np.float32)
    return z["lon"],z["lat"],feat,sp

# ---------- baseline gate ----------
completion_rate = float(baseline_report.completed.mean()) if len(baseline_report) else 0.0

tier_a = baseline_report[
    baseline_report.completed &
    (baseline_report.loop_events.fillna(99) == 0)
].copy()

# 对转角使用自适应阈值，避免硬编码过紧
completed_turns = baseline_report.loc[
    baseline_report.completed, "p95_turn_deg"
].dropna()

if len(completed_turns):
    turn_limit_b = max(60.0, float(completed_turns.quantile(0.80)))
    turn_limit_c = max(90.0, float(completed_turns.quantile(0.95)))
else:
    turn_limit_b = 90.0
    turn_limit_c = 120.0

tier_b = baseline_report[
    baseline_report.completed &
    (baseline_report.loop_events.fillna(99) <= 1) &
    (
        baseline_report.p95_turn_deg.isna() |
        (baseline_report.p95_turn_deg <= turn_limit_b)
    )
].copy()

tier_c = baseline_report[
    baseline_report.completed &
    (baseline_report.loop_events.fillna(99) <= 2) &
    (
        baseline_report.p95_turn_deg.isna() |
        (baseline_report.p95_turn_deg <= turn_limit_c)
    )
].copy()

# 质量排序：少loop、较低p95转角、较短异常路径优先
if len(tier_c):
    med_path = tier_c.path_km.median()
    tier_c["quality_score"] = (
        tier_c.loop_events.fillna(9)*10.0 +
        tier_c.p95_turn_deg.fillna(180.0)/30.0 +
        np.abs(tier_c.path_km.fillna(med_path)-med_path)/(med_path+1e-6)
    )
    tier_c = tier_c.sort_values(["quality_score","date"])

print("completion rate:", completion_rate)
print("Tier A days:", len(tier_a))
print("Tier B days:", len(tier_b), "turn_limit_b=", turn_limit_b)
print("Tier C days:", len(tier_c), "turn_limit_c=", turn_limit_c)

if completion_rate < cfg.MIN_COMPLETION_RATE_FOR_CNN:
    print(
        f"[WARN] baseline completion={completion_rate:.3f} < "
        f"{cfg.MIN_COMPLETION_RATE_FOR_CNN:.3f}. "
        "这说明tracker仍需要重点检查。"
    )

if len(tier_a) >= cfg.MIN_WEAK_LABEL_DAYS:
    eligible = tier_a
    weak_label_tier = "A_completed_zero_loop"
elif len(tier_b) >= cfg.MIN_WEAK_LABEL_DAYS:
    eligible = tier_b
    weak_label_tier = "B_completed_le1_loop_geometry_filtered"
elif len(tier_c) >= cfg.MIN_WEAK_LABEL_DAYS:
    eligible = tier_c
    weak_label_tier = "C_completed_le2_loop_geometry_filtered"
else:
    diag_cols = [
        "date","completed","loop_events","n_points",
        "max_lon","final_lon","stop_reason","p95_turn_deg"
    ]
    display(
        baseline_report.sort_values(
            ["completed","max_lon"],
            ascending=[False,False]
        )[diag_cols].head(40)
    )
    raise RuntimeError(
        "FIX2 Gate: baseline没有足够的完整高质量日期来训练Anti-Eddy CNN。"
        f" TierA={len(tier_a)}, TierB={len(tier_b)}, TierC={len(tier_c)}, "
        f"最低要求={cfg.MIN_WEAK_LABEL_DAYS}。"
        " 请把 reports/baseline_failure_diagnostic_FIX2.csv 回传分析；"
        "不要通过放宽到“不完整路径”来强行训练CNN。"
    )

print("weak-label tier selected:", weak_label_tier)
print("eligible days:", len(eligible))

eligible[[
    "date","completed","loop_events","n_points",
    "path_km","p95_turn_deg","max_lon","stop_reason"
]].to_csv(
    DIRS["reports"]/"weak_label_eligible_days_FIX2.csv",
    index=False,
    encoding="utf-8-sig"
)

# ---------- patch生成 ----------
X_list, y_list, d_list = [], [], []

for rr in tqdm(
    eligible.itertuples(index=False),
    total=len(eligible),
    desc=f"patch dataset [{weak_label_tier}]"
):
    dt = pd.Timestamp(rr.date)
    tr = load_track_npz(
        DIRS["tracks_base"] / f"{dt:%Y%m%d}.npz"
    )

    lon_g, lat_g, feat, speed = day_features(dt)
    ui_speed = RegularGridInterpolator(
        (lat_g,lon_g),
        speed,
        bounds_error=False,
        fill_value=np.nan
    )

    idxs = np.arange(
        2,
        max(2,len(tr["lon"])-2),
        cfg.PATCH_STRIDE_ALONG_TRACK
    )

    if len(idxs) == 0:
        continue

    if len(idxs) > cfg.MAX_POS_PER_DAY:
        idxs = np.linspace(
            idxs[0],
            idxs[-1],
            cfg.MAX_POS_PER_DAY
        ).astype(int)

    for i in idxs:
        lon0 = float(tr["lon"][i])
        lat0 = float(tr["lat"][i])

        p = extract_patch(feat,lon_g,lat_g,lon0,lat0)
        if p is None:
            continue

        X_list.append(p)
        y_list.append(1)
        d_list.append(dt)

        # FIX2：这里也统一使用东向=0的角度
        tangent = angle_east_rad(
            float(tr["lon"][i-1]),
            float(tr["lat"][i-1]),
            float(tr["lon"][i+1]),
            float(tr["lat"][i+1])
        )

        candidate_offsets = [30,40,50,60,70,80]
        nneg = min(cfg.NEG_PER_POS, len(candidate_offsets))
        offsets = random.sample(candidate_offsets, k=nneg)

        # 尽量左右各取一个
        signs = [1,-1] if nneg >= 2 else [random.choice([1,-1])]
        signs = signs[:nneg]

        for off,sgn in zip(offsets,signs):
            nl,na = forward_point(
                lon0,lat0,
                tangent + sgn*np.pi/2,
                off
            )

            npatch = extract_patch(feat,lon_g,lat_g,nl,na)
            if npatch is None:
                continue

            pos_speed = float(ui_speed([[lat0,lon0]])[0])
            neg_speed = float(ui_speed([[na,nl]])[0])

            # hard negative：邻近区域流速必须有一定竞争性
            if (
                np.isfinite(neg_speed) and
                np.isfinite(pos_speed) and
                neg_speed >= 0.45*pos_speed
            ):
                X_list.append(npatch)
                y_list.append(0)
                d_list.append(dt)

print("patches collected:", len(X_list))

if len(X_list) == 0:
    raise RuntimeError(
        "FIX2 patch Gate: eligible路径存在，但没有成功提取任何patch。"
        " 请检查 PATCH_SIZE、研究区边界、finite比例和50m cache。"
    )

X = np.stack(X_list).astype(np.float32)
y = np.asarray(y_list,dtype=np.float32)
dates = pd.to_datetime(pd.Series(d_list)).to_numpy()

if len(np.unique(y)) < 2:
    raise RuntimeError(
        f"FIX2 patch Gate: 只有单一类别，positive ratio={y.mean():.3f}。"
        " CNN分类训练需要同时存在正样本和hard-negative。"
    )

n_unique_dates = len(np.unique(dates.astype("datetime64[D]")))
if n_unique_dates < 6:
    raise RuntimeError(
        f"FIX2 patch Gate: 有patch的独立日期仅 {n_unique_dates} 天，"
        "无法可靠分train/val/test。"
    )

print("X:",X.shape)
print("positive ratio:",float(y.mean()))
print("unique dates:",n_unique_dates)
print("date range:",dates.min(),"->",dates.max())

patch_path = DIRS["dataset"]/"anti_eddy_patches_FIX2.npz"
np.savez_compressed(
    patch_path,
    X=X,
    y=y,
    dates=dates.astype("datetime64[D]").astype(str),
    weak_label_tier=np.array(weak_label_tier)
)
print("saved:",patch_path)


In [ ]:

# ===== 10. 日期分块 split + TinyJetCNN 训练 — FIX2 =====
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =",device)

unique_dates = np.array(
    sorted(pd.to_datetime(pd.Series(dates)).dt.normalize().unique())
)
n = len(unique_dates)

# 至少给train/val/test各2个日期
train_cut = max(2, int(n*0.70))
val_cut = max(train_cut+2, int(n*0.85))
val_cut = min(val_cut, n-2)

d_train_end = unique_dates[train_cut-1]
d_val_end = unique_dates[val_cut-1]

dates_pd = pd.to_datetime(dates)
m_train = dates_pd <= d_train_end
m_val = (dates_pd > d_train_end) & (dates_pd <= d_val_end)
m_test = dates_pd > d_val_end

print(
    "date blocks:",
    unique_dates[0],"->",d_train_end,
    "|",d_val_end,
    "|",unique_dates[-1]
)
print(
    "samples:",
    int(m_train.sum()),
    int(m_val.sum()),
    int(m_test.sum())
)

for name,mask in [("train",m_train),("val",m_val),("test",m_test)]:
    yy = y[mask]
    print(
        name,
        "N=",len(yy),
        "pos_ratio=",float(yy.mean()) if len(yy) else np.nan,
        "classes=",np.unique(yy)
    )
    if len(yy)==0 or len(np.unique(yy))<2:
        raise RuntimeError(
            f"{name} block缺少样本或只有单一类别。"
            " 请增加weak-label日期或hard-negative数量。"
        )

# normalization只用train
ch_mean = X[m_train].mean(axis=(0,2,3))
ch_std = X[m_train].std(axis=(0,2,3)) + 1e-6
Xn = (
    X - ch_mean[None,:,None,None]
) / ch_std[None,:,None,None]

class TinyJetCNN(nn.Module):
    def __init__(self,cin=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin,16,3,padding=1),
            nn.BatchNorm2d(16),
            nn.SiLU(),

            nn.Conv2d(16,32,3,stride=2,padding=1),
            nn.BatchNorm2d(32),
            nn.SiLU(),

            nn.Conv2d(32,64,3,stride=2,padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64,32),
            nn.SiLU(),
            nn.Dropout(0.20),
            nn.Linear(32,1),
        )

    def forward(self,x):
        return self.net(x).squeeze(-1)

def loader(mask,shuffle=False):
    xx = torch.from_numpy(Xn[mask]).float()
    yy = torch.from_numpy(y[mask]).float()
    return DataLoader(
        TensorDataset(xx,yy),
        batch_size=cfg.CNN_BATCH,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

train_loader = loader(m_train,True)
val_loader = loader(m_val,False)
test_loader = loader(m_test,False)

model = TinyJetCNN(cfg.PATCH_CHANNELS).to(device)

# 弱监督样本可能正负不平衡，使用train-only pos_weight
n_pos = float(y[m_train].sum())
n_neg = float(m_train.sum()-n_pos)
pos_weight = torch.tensor(
    [max(n_neg/max(n_pos,1.0), 0.1)],
    dtype=torch.float32,
    device=device
)

crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.CNN_LR,
    weight_decay=1e-4
)

best = {"loss":np.inf,"state":None,"epoch":-1}
history = []

for epoch in range(1,cfg.CNN_EPOCHS+1):
    model.train()
    tr_loss,nobs = 0.0,0

    for xb,yb in train_loader:
        xb = xb.to(device,non_blocking=True)
        yb = yb.to(device,non_blocking=True)

        opt.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = crit(logits,yb)
        loss.backward()
        opt.step()

        tr_loss += float(loss)*len(xb)
        nobs += len(xb)

    tr_loss /= max(nobs,1)

    model.eval()
    vals,vys = [],[]
    vl,vn = 0.0,0

    with torch.no_grad():
        for xb,yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = crit(logits,yb)

            vl += float(loss)*len(xb)
            vn += len(xb)
            vals.append(torch.sigmoid(logits).cpu().numpy())
            vys.append(yb.cpu().numpy())

    vloss = vl/max(vn,1)
    vp = np.concatenate(vals)
    vy = np.concatenate(vys)
    vauc = roc_auc_score(vy,vp)

    history.append((epoch,tr_loss,vloss,vauc))

    print(
        f"epoch={epoch:02d} "
        f"train={tr_loss:.4f} "
        f"val={vloss:.4f} "
        f"val_auc={vauc:.4f}"
    )

    if vloss < best["loss"]:
        best["loss"] = vloss
        best["epoch"] = epoch
        best["state"] = {
            k:v.detach().cpu().clone()
            for k,v in model.state_dict().items()
        }

if best["state"] is None:
    raise RuntimeError("CNN训练未产生有效best checkpoint")

model.load_state_dict(best["state"])
model.eval()

tp,ty = [],[]
with torch.no_grad():
    for xb,yb in test_loader:
        p = torch.sigmoid(
            model(xb.to(device))
        ).cpu().numpy()
        tp.append(p)
        ty.append(yb.numpy())

tp = np.concatenate(tp)
ty = np.concatenate(ty)
pred = (tp>=0.5).astype(int)

auc = roc_auc_score(ty,tp)
acc = accuracy_score(ty,pred)
pr,rc,f1,_ = precision_recall_fscore_support(
    ty,pred,
    average="binary",
    zero_division=0
)

print({
    "test_auc":float(auc),
    "acc":float(acc),
    "precision":float(pr),
    "recall":float(rc),
    "f1":float(f1),
    "best_epoch":best["epoch"],
    "weak_label_tier":weak_label_tier,
})

# FIX3: checkpoint只保存weights_only安全类型
# NumPy数组先转成torch.Tensor，避免PyTorch 2.6+安全加载失败
ckpt = {
    "model_state": model.state_dict(),
    "channel_mean": torch.as_tensor(ch_mean, dtype=torch.float32).cpu(),
    "channel_std": torch.as_tensor(ch_std, dtype=torch.float32).cpu(),
    "best_epoch": int(best["epoch"]),
    "weak_label_tier": str(weak_label_tier),
    "tracker_version": str(cfg.TRACKER_VERSION),
}

safe_ckpt_path = DIRS["models"]/"anti_eddy_cnn_FIX3_safe.pt"
torch.save(ckpt, safe_ckpt_path)

# 配置元数据单独写JSON，避免把复杂Python/NumPy对象塞进torch checkpoint
with open(DIRS["models"]/"anti_eddy_cnn_FIX3_meta.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "cfg": asdict(cfg),
            "best_epoch": int(best["epoch"]),
            "weak_label_tier": str(weak_label_tier),
            "tracker_version": str(cfg.TRACKER_VERSION),
        },
        f,
        indent=2,
        ensure_ascii=False,
        default=list,
    )

pd.DataFrame(
    history,
    columns=["epoch","train_loss","val_loss","val_auc"]
).to_csv(
    DIRS["reports"]/"cnn_history_FIX2.csv",
    index=False
)

print("saved safe model:", safe_ckpt_path)


In [ ]:

# ===== 10.5 FIX3 checkpoint安全加载 / 旧FIX2一次性迁移 =====
from pathlib import Path
import numpy as np
import torch

SAFE_CKPT = DIRS["models"] / "anti_eddy_cnn_FIX3_safe.pt"
OLD_FIX2_CKPT = DIRS["models"] / "anti_eddy_cnn_FIX2.pt"

def _convert_old_ckpt_to_safe(old_ckpt, safe_path):
    """
    仅用于迁移“你自己在本机用本Notebook生成的FIX2 checkpoint”。

    PyTorch 2.6+默认weights_only=True。
    旧FIX2 checkpoint含NumPy数组，所以安全加载失败。
    这里一次性使用weights_only=False读取可信的本地旧文件，
    立即转存为只含Tensor/primitive的安全checkpoint。
    """
    print("[FIX3] 正在迁移可信的本地 FIX2 checkpoint:", old_ckpt)

    old = torch.load(
        old_ckpt,
        map_location="cpu",
        weights_only=False,   # 仅对自己本地生成的旧FIX2文件使用一次
    )

    safe = {
        "model_state": old["model_state"],
        "channel_mean": torch.as_tensor(
            np.asarray(old["channel_mean"]),
            dtype=torch.float32
        ).cpu(),
        "channel_std": torch.as_tensor(
            np.asarray(old["channel_std"]),
            dtype=torch.float32
        ).cpu(),
        "best_epoch": int(old.get("best_epoch", -1)),
        "weak_label_tier": str(old.get("weak_label_tier", "unknown")),
        "tracker_version": str(
            old.get("tracker_version", cfg.TRACKER_VERSION)
        ),
    }

    torch.save(safe, safe_path)
    print("[FIX3] 迁移完成 ->", safe_path)

def load_safe_cnn_checkpoint():
    # 优先读取FIX3安全checkpoint
    if SAFE_CKPT.exists():
        print("[FIX3] loading safe checkpoint:", SAFE_CKPT)
        return torch.load(
            SAFE_CKPT,
            map_location="cpu",
            weights_only=True,
        )

    # 如果只存在旧FIX2 checkpoint，则执行一次迁移
    if OLD_FIX2_CKPT.exists():
        _convert_old_ckpt_to_safe(
            OLD_FIX2_CKPT,
            SAFE_CKPT
        )
        return torch.load(
            SAFE_CKPT,
            map_location="cpu",
            weights_only=True,
        )

    raise FileNotFoundError(
        "未找到 CNN checkpoint。\n"
        f"安全FIX3: {SAFE_CKPT}\n"
        f"旧FIX2: {OLD_FIX2_CKPT}\n"
        "请先运行CNN训练单元。"
    )

# 先做一次加载smoke test
_ckpt_smoke = load_safe_cnn_checkpoint()

required_keys = {
    "model_state",
    "channel_mean",
    "channel_std",
    "best_epoch",
    "weak_label_tier",
}
missing = required_keys - set(_ckpt_smoke.keys())
if missing:
    raise RuntimeError(
        f"checkpoint缺少字段: {sorted(missing)}"
    )

assert torch.is_tensor(_ckpt_smoke["channel_mean"])
assert torch.is_tensor(_ckpt_smoke["channel_std"])

print("[FIX3 checkpoint smoke] PASS")
print("best_epoch =", _ckpt_smoke["best_epoch"])
print("weak_label_tier =", _ckpt_smoke["weak_label_tier"])
print("channel_mean shape =", tuple(_ckpt_smoke["channel_mean"].shape))


In [ ]:

# ===== 11. 用 CNN-head 重跑 tracker，并与 baseline 做资格对比 =====
ckpt = load_safe_cnn_checkpoint()
model2 = TinyJetCNN(cfg.PATCH_CHANNELS)
model2.load_state_dict(ckpt["model_state"])
model2 = model2.to(device).eval()
cnn_bundle = (
    model2,
    ckpt["channel_mean"].detach().cpu().numpy().astype(np.float32),
    ckpt["channel_std"].detach().cpu().numpy().astype(np.float32),
    device
)

cnn_rows = []
for row in tqdm(cache_ok.itertuples(index=False), total=len(cache_ok), desc="cnn-head axis"):
    dt = pd.Timestamp(row.date)
    out = DIRS["tracks_cnn"] / f"{dt:%Y%m%d}.npz"
    try:
        if out.exists():
            tr = load_track_npz(out)
            st = "cached"
        else:
            z = np.load(DIRS["cache50"] / f"{dt:%Y%m%d}.npz")
            tr = track_axis(z["lon"], z["lat"], z["u"], z["v"], cnn_bundle=cnn_bundle)
            save_track(tr, out)
            st = "written"
        gm = track_geometry_metrics(tr)
        cnn_rows.append({"date":dt,"status":st,"completed":tr["completed"],
                         "loop_events":tr["loop_events"],"n_points":tr["n_points"],**gm})
    except Exception as e:
        cnn_rows.append({"date":dt,"status":"error:"+repr(e),"completed":False,
                         "loop_events":np.nan,"n_points":0,
                         "path_km":np.nan,"p95_turn_deg":np.nan,"max_turn_deg":np.nan})

cnn_report = pd.DataFrame(cnn_rows).sort_values("date")
cnn_report.to_csv(DIRS["reports"]/"cnn_tracker_report_FIX2.csv", index=False, encoding="utf-8-sig")

cmp = baseline_report.merge(cnn_report, on="date", suffixes=("_base","_cnn"))
summary = pd.DataFrame({
    "metric": ["completion_rate","loop_day_rate","median_path_km","median_p95_turn_deg"],
    "baseline": [
        cmp.completed_base.mean(),
        (cmp.loop_events_base.fillna(0)>0).mean(),
        cmp.path_km_base.median(),
        cmp.p95_turn_deg_base.median(),
    ],
    "cnn_head": [
        cmp.completed_cnn.mean(),
        (cmp.loop_events_cnn.fillna(0)>0).mean(),
        cmp.path_km_cnn.median(),
        cmp.p95_turn_deg_cnn.median(),
    ]
})
display(summary)
summary.to_csv(DIRS["reports"]/"tracker_qualification_summary_FIX2.csv", index=False)

# 给可疑日期一个排序分数，优先人工检查
cmp["suspicion_base"] = (
    (~cmp.completed_base).astype(float)*5 +
    cmp.loop_events_base.fillna(0)*3 +
    (cmp.p95_turn_deg_base.fillna(0)/90.0)
)
cmp["suspicion_cnn"] = (
    (~cmp.completed_cnn).astype(float)*5 +
    cmp.loop_events_cnn.fillna(0)*3 +
    (cmp.p95_turn_deg_cnn.fillna(0)/90.0)
)
cmp["manual_priority"] = np.maximum(cmp.suspicion_base, cmp.suspicion_cnn)
worst = cmp.sort_values("manual_priority", ascending=False).head(24)
display(worst[["date","completed_base","completed_cnn","loop_events_base","loop_events_cnn",
               "p95_turn_deg_base","p95_turn_deg_cnn","manual_priority"]])


In [ ]:

# ===== 12. 自动画最可疑日期：baseline vs CNN-head + 速度场 =====
import matplotlib.pyplot as plt

plot_dir = DIRS["figures"]/"manual_qc"
plot_dir.mkdir(parents=True, exist_ok=True)

def plot_day_compare(dt):
    dt = pd.Timestamp(dt)
    z = np.load(DIRS["cache50"]/f"{dt:%Y%m%d}.npz")
    b = load_track_npz(DIRS["tracks_base"]/f"{dt:%Y%m%d}.npz")
    c = load_track_npz(DIRS["tracks_cnn"]/f"{dt:%Y%m%d}.npz")
    sp = np.hypot(z["u"], z["v"])

    fig, ax = plt.subplots(figsize=(10,7))
    im = ax.pcolormesh(z["lon"], z["lat"], sp, shading="auto")
    ax.plot(b["lon"], b["lat"], lw=1.2, label=f"baseline loops={b['loop_events']}")
    ax.plot(c["lon"], c["lat"], lw=1.2, label=f"CNN-head loops={c['loop_events']}")
    ax.set_xlim(120,147); ax.set_ylim(20,42)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_title(str(dt.date()))
    ax.legend()
    fig.colorbar(im, ax=ax, label="speed (m/s)")
    fig.tight_layout()
    out = plot_dir/f"{dt:%Y%m%d}_compare.png"
    fig.savefig(out, dpi=180)
    plt.close(fig)
    return out

qc_rows = []
for dt in worst.date:
    out = plot_day_compare(dt)
    qc_rows.append({
        "date": pd.Timestamp(dt).date().isoformat(),
        "figure": str(out),
        "baseline_ok": "",
        "cnn_ok": "",
        "preferred": "",  # baseline / cnn / neither
        "notes": ""
    })

qc = pd.DataFrame(qc_rows)
qc_path = DIRS["reports"]/"manual_qc_template.csv"
qc.to_csv(qc_path, index=False, encoding="utf-8-sig")
print("人工QC模板:", qc_path)
print("图片目录:", plot_dir)
display(qc.head())



## 13. 双分支逐日路径表示

保持原论文核心表示不变：

- **Branch 1**：22°N–25.5°N，固定纬度，以经度表示路径；
- **Branch 2**：122°E–145°E，固定经度，以纬度表示路径；
- 目标网格间隔仍为 1/12°。

下面尽量复现原 MATLAB 逻辑，并把日期作为第一主键，不再用“第几月/第几列”假设。


In [ ]:

# ===== 13. 从 CNN-head 路径构建逐日双分支数据集 =====
from scipy.interpolate import interp1d

TARGET_LATS_1 = np.arange(22.0, 25.5 + 1e-9, 1/12)
TARGET_LONS_2 = np.arange(122.0, 145.0 + 1e-9, 1/12)

def branch1_from_track(lons, lats):
    valid = np.isfinite(lons) & np.isfinite(lats) & (lats <= 25.7)
    lons, lats = np.asarray(lons)[valid], np.asarray(lats)[valid]
    if len(lats) < 2:
        return np.full_like(TARGET_LATS_1, np.nan, dtype=float)
    # 重复纬度取平均经度
    df = pd.DataFrame({"lat":lats,"lon":lons}).groupby("lat",as_index=False).lon.mean().sort_values("lat")
    if len(df)<2:
        return np.full_like(TARGET_LATS_1, np.nan, dtype=float)
    f = interp1d(df.lat.values, df.lon.values, kind="linear", bounds_error=False, fill_value="extrapolate")
    out = f(TARGET_LATS_1)
    min_data_lat = df.lat.min()
    ext = TARGET_LATS_1 < min_data_lat
    if ext.any() and np.any(out[ext] < 121.0):
        start_lon = float(f(min_data_lat))
        dist = min_data_lat - TARGET_LATS_1[ext]
        den = max(min_data_lat - TARGET_LATS_1.min(), 1e-9)
        out[ext] = start_lon + (121.0-start_lon)*(dist/den)
    return out.astype(np.float32)

def branch2_from_track(lons, lats):
    valid = np.isfinite(lons) & np.isfinite(lats)
    lons, lats = np.asarray(lons)[valid], np.asarray(lats)[valid]
    if len(lons)<2:
        return np.full_like(TARGET_LONS_2, np.nan, dtype=float)
    # 复现原逻辑：lon<=125 取最后/最高分支，lon>125 取平均
    vals = []
    for lo in np.sort(np.unique(lons)):
        latm = lats[lons==lo]
        vals.append((lo, np.max(latm) if lo<=125 else np.mean(latm)))
    arr = np.asarray(vals, dtype=float)
    slon, slat = arr[:,0], arr[:,1]
    if len(slon)<2:
        return np.full_like(TARGET_LONS_2, np.nan, dtype=float)

    out = np.full_like(TARGET_LONS_2, np.nan, dtype=float)
    # 第一点：沿用旧代码实际行为，右侧使用最小纬度（源代码注释与实现不一致，后续可做消融）
    cur = TARGET_LONS_2[0]
    left = slon <= cur; right = slon >= cur
    if left.any() and right.any():
        il = np.argmax(slat[left])
        lons_l, lats_l = slon[left], slat[left]
        ir = np.argmin(slat[right])
        lons_r, lats_r = slon[right], slat[right]
        x0,x1 = lons_l[il],lons_r[ir]
        y0,y1 = lats_l[il],lats_r[ir]
        if abs(x1-x0)>1e-12:
            out[0] = y0 + (y1-y0)*(cur-x0)/(x1-x0)
    if not np.isfinite(out[0]):
        f0 = interp1d(slon,slat,bounds_error=False,fill_value="extrapolate")
        out[0] = float(f0(cur))

    # 后续点：选择当前经度右侧、与上一插值点欧氏距离最近的原始点
    for i in range(1,len(TARGET_LONS_2)):
        cur = TARGET_LONS_2[i]
        prev_lon, prev_lat = TARGET_LONS_2[i-1], out[i-1]
        gr = slon > cur
        if not gr.any():
            f0 = interp1d(slon,slat,bounds_error=False,fill_value="extrapolate")
            out[i] = float(f0(cur))
            continue
        cand_lon, cand_lat = slon[gr], slat[gr]
        dist = np.hypot(cand_lat-prev_lat, cand_lon-prev_lon)
        k = int(np.argmin(dist))
        x1,y1 = cand_lon[k],cand_lat[k]
        if abs(x1-prev_lon)<1e-12:
            out[i] = y1
        else:
            out[i] = prev_lat + (y1-prev_lat)*(cur-prev_lon)/(x1-prev_lon)
    return out.astype(np.float32)

date_list, b1_list, b2_list = [], [], []
for dt in tqdm(cache_ok.date, desc="dual branch"):
    dt = pd.Timestamp(dt)
    p = DIRS["tracks_cnn"]/f"{dt:%Y%m%d}.npz"
    if not p.exists():
        continue
    tr = load_track_npz(p)
    if not tr["completed"]:
        continue
    b1 = branch1_from_track(tr["lon"],tr["lat"])
    b2 = branch2_from_track(tr["lon"],tr["lat"])
    date_list.append(dt)
    b1_list.append(b1); b2_list.append(b2)

B1 = np.stack(b1_list).astype(np.float32)  # [time, 43]
B2 = np.stack(b2_list).astype(np.float32)  # [time, 277]
D = pd.to_datetime(date_list)

np.savez_compressed(
    DIRS["dataset"]/"daily_dual_branch_paths.npz",
    dates=np.array([d.strftime("%Y-%m-%d") for d in D]),
    branch1_lon=B1, branch1_target_lats=TARGET_LATS_1,
    branch2_lat=B2, branch2_target_lons=TARGET_LONS_2
)

meta = pd.DataFrame({
    "date": D,
    "branch1_nan_frac": np.isnan(B1).mean(axis=1),
    "branch2_nan_frac": np.isnan(B2).mean(axis=1),
})
meta.to_csv(DIRS["dataset"]/"daily_dual_branch_meta.csv",index=False,encoding="utf-8-sig")

print("dates:", len(D), D.min(), D.max())
print("B1:", B1.shape, "B2:", B2.shape)
display(meta.describe())



## 14. 本轮 Gate（不要跳过）

满足以下条件后，才建议进入正式的 EOF-TCN-LSTM / GRU 日尺度预测训练：

1. NetCDF 日期审计完成，缺失/坏文件都有 manifest；
2. baseline 与 CNN-head 至少都能在绝大多数可用日完整到达 146°E；
3. CNN-head 的 loop-day rate 不高于 baseline，且人工 QC 的可疑样本中没有系统性“走错主流轴”；
4. 双分支矩阵中的 NaN 比例可控；
5. 人工 QC 至少完成一批强涡/急弯/KE 区域难例；
6. **最终预测模型必须使用 blocked/rolling 时间划分，scaler 与 EOF 只在 train block 拟合。**

> 当前上传文件中没有实际的 LSTM/GRU/TCN 网络训练源码。  
> 因此这个 Notebook 在“数据 + 抗涡识别 + 双分支数据集”处冻结；下一份预测 Notebook 应在拿到原模型实现后按原网络精确改造成 daily 版本，避免凭论文描述重新猜网络。


In [ ]:

# ===== 15. 生成本轮 handoff 摘要 =====
handoff = {
    "cfg": asdict(cfg),
    "raw_days_expected": int(len(manifest)),
    "raw_days_present": int((manifest.raw_status=="present").sum()),
    "raw_days_missing": int((manifest.raw_status=="missing").sum()),
    "baseline_completion": float(baseline_report.completed.mean()) if len(baseline_report) else None,
    "cnn_completion": float(cnn_report.completed.mean()) if 'cnn_report' in globals() and len(cnn_report) else None,
    "baseline_loop_day_rate": float((baseline_report.loop_events.fillna(0)>0).mean()) if len(baseline_report) else None,
    "cnn_loop_day_rate": float((cnn_report.loop_events.fillna(0)>0).mean()) if 'cnn_report' in globals() and len(cnn_report) else None,
    "dual_branch_days": int(len(D)) if 'D' in globals() else 0,
    "artifacts": {k:str(v) for k,v in DIRS.items()},
    "tracker_version": cfg.TRACKER_VERSION,
    "weak_label_tier": weak_label_tier if "weak_label_tier" in globals() else None,
    "note": "R1 FIX2 qualification only; angle convention repaired. Independent physical-axis validation and final temporal model training remain."
}
with open(DIRS["reports"]/"R1_FIX2_HANDOFF.json","w",encoding="utf-8") as f:
    json.dump(handoff,f,indent=2,ensure_ascii=False,default=str)
print(json.dumps(handoff,indent=2,ensure_ascii=False,default=str))


## FIX1 路径说明

```text
原始数据：
E:\\2023\\
E:\\2024\\
E:\\2025\\
        ↓  WSL
/mnt/e/2023
/mnt/e/2024
/mnt/e/2025

处理缓存与实验结果：
D:\\KuroshioDailyR1\\
        ↓  WSL
/mnt/d/KuroshioDailyR1/
```

外置 USB 只承担顺序读取原始 NetCDF；cache、路径、模型和报告都写内部 D 盘。
如果 E 盘暂时断开后重新插入，只要 Windows 仍分配为 `E:`，重新运行时 D 盘已成功缓存的日期会显示为 `cached`，不会重复处理。



## FIX2 重跑说明

如果你已经完成了 NetCDF → `cache_50m`：

- **不需要重新处理 E 盘 NetCDF**
- 保留 `D:\KuroshioDailyR1\cache_50m`
- 从本 Notebook 的 **第 6 节“几何与物理特征函数”开始重新运行即可**

FIX2 会写入新的：

```text
tracks_baseline_FIX2_east0/
tracks_cnn_FIX2_east0/
models/anti_eddy_cnn_FIX2.pt
reports/baseline_tracker_report_FIX2.csv
reports/baseline_failure_diagnostic_FIX2.csv
```

因此不会误用 FIX1 的错误轨迹缓存。



## FIX3：你当前已经训练完 FIX2 CNN 时怎么继续

你**不需要重新训练 CNN**。

如果目录中已经有：

```text
D:\KuroshioDailyR1\models\anti_eddy_cnn_FIX2.pt
```

直接执行：

```text
第 10.5 节 FIX3 checkpoint安全加载 / 旧FIX2一次性迁移
```

它会产生：

```text
anti_eddy_cnn_FIX3_safe.pt
```

然后继续执行：

```text
第 11 节 用 CNN-head 重跑 tracker
```

即可。

只有旧 checkpoint 不是你自己本地生成、来源不可信时，才不要使用迁移函数里的 `weights_only=False`。
